In [1]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
from src import interpretation as interpret

/Users/hector.vargas/repos/ml_hands_on_project/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
X_train = pd.read_csv("../data/processed/raw_features/X_train.csv")
X_test = pd.read_csv("../data/processed/raw_features/X_test.csv")

ROOT_DIR = os.path.abspath("../")
DB_PATH = os.path.join(ROOT_DIR, "mlflow.db")
EXPERIMENT_NAME = "customer-churn-optuna"

In [3]:
FEATURE_REGISTRY_BINARY = {
    # --- Binary Flags ---
    "is_silver": lambda X: X["Card Type"] == 'SILVER',
    "is_germany": lambda X: X["Geography"] == 'Germany',
    "is_spain": lambda X: X["Geography"] == 'Spain',
    "Num_Of_Products_1": lambda X: X["NumOfProducts"] == 1,
    "Num_Of_Products_2": lambda X: X["NumOfProducts"] == 2,
    "Num_Of_Products_3": lambda X: X["NumOfProducts"] == 3,
    "Num_Of_Products_4": lambda X: X["NumOfProducts"] == 4,
}

FEATURE_REGISTRY_CONTINUOUS = {
    # --- Polynomial & Interaction Terms ---
    "Age_x_IsActive": lambda X: X["Age"] * X["IsActiveMember"],

    # --- Financial & Engagement Ratios ---
    "Balance_per_Product": lambda X: X["Balance"] / (X["NumOfProducts"] + 1),
    "CreditScore_per_Age": lambda X: X["CreditScore"] / (X["Age"] + 1),

    # --- Behavioral Cross-Products ---
    "Inactive_x_Balance": lambda X: (1 - X["IsActiveMember"]) * X["Balance"],

    # --- Monetary Accumulations & Non-linear Scaling ---
    "CreditScore_x_Age": lambda X: X["CreditScore"] * X["Age"],

    # --- Temporal Product Densities ---
    "Products_per_Tenure": lambda X: X["NumOfProducts"] / (X["Tenure"] + 1),
}
# Schema Baseline Columns Definitions
nomod_columns = []

dummyfy_columns = [
    'Gender'
    ]

norm_std_columns = [
    'Point Earned',
    'Satisfaction Score', 
    'EstimatedSalary'
    ]

In [4]:
all_engineered_features = list(FEATURE_REGISTRY_BINARY.keys())

EXPERIMENT_REGISTRY = {
    # Experiment 2: 
    "experiment_2": {
        "passthrough": nomod_columns + list(FEATURE_REGISTRY_BINARY.keys()),
        "standard_scale": norm_std_columns + list(FEATURE_REGISTRY_CONTINUOUS.keys()),
        "one_hot_encode": dummyfy_columns
    },
}
# Combine registries purely for execution lookup inside the transformer
FULL_REGISTRY = {**FEATURE_REGISTRY_BINARY, **FEATURE_REGISTRY_CONTINUOUS}
current_layout = EXPERIMENT_REGISTRY['experiment_2']

In [ ]:
def extract_pipeline_components(pipeline):
    """
    Splits the fitted sklearn pipeline into its major stages.
    """
    fe_step = pipeline.named_steps["feature_engineering"]
    preprocessor = pipeline.named_steps["preprocessing"]
    model = pipeline.named_steps["model"]

    return fe_step, preprocessor, model

def print_model_details(best_run, pipeline):
    """
    Prints MLflow metadata and model hyperparameters.
    """

    print("\n========================")
    print("BEST RUN")
    print("========================")

    print(f"Run ID: {best_run.run_id}")

    if "metrics.average_precision" in best_run:
        print(
            f"Average Precision: "
            f"{best_run['metrics.average_precision']:.5f}"
        )

    model = pipeline.named_steps["model"]

    print("\n========================")
    print("MODEL PARAMETERS")
    print("========================")

    for k, v in model.get_params().items():
        print(f"{k}: {v}")

In [ ]:
def transform_raw_data(
    pipeline,
    X_raw: pd.DataFrame
):
    """
    Applies feature engineering + preprocessing.
    Returns transformed dataframe with labels.
    """

    fe_step = pipeline.named_steps["feature_engineering"]
    preprocessor = pipeline.named_steps["preprocessing"]

    X_engineered = fe_step.transform(X_raw)

    X_transformed = preprocessor.transform(X_engineered)

    feature_names = (
        preprocessor.get_feature_names_out()
    )

    X_transformed_df = pd.DataFrame(
        X_transformed,
        columns=feature_names,
        index=X_raw.index,
    )

    return X_transformed_df

In [ ]:
def predict_customer_churn(
    pipeline,
    X_raw: pd.DataFrame,
    threshold: float = 0.5,
):
    """
    Predict churn probability from raw customer data.
    """

    probabilities = pipeline.predict_proba(X_raw)[:, 1]

    predictions = (
        probabilities >= threshold
    ).astype(int)

    return pd.DataFrame(
        {
            "churn_probability": probabilities,
            "prediction": predictions,
        },
        index=X_raw.index,
    )

In [ ]:
import mlflow

experiment = mlflow.get_experiment_by_name(
    "customer-churn-optuna"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.average_precision DESC"]
)

runs.head()

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.average_precision,metrics.train_auc,metrics.train_precision,metrics.test_recall,...,tags.number,tags.datetime_complete,tags.xgb_colsample_bytree_distribution,tags.xgb_reg_alpha_distribution,tags.xgb_subsample_distribution,tags.mlflow.runName,tags.rf_max_depth_distribution,tags.rf_min_samples_split_distribution,tags.rf_n_estimators_distribution,tags.rf_min_samples_leaf_distribution
0,6c5042b4d21c4a1bb11d30c685708b5b,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-17 23:12:33.816000+00:00,2026-06-17 23:12:33.828000+00:00,0.697040,NaN,NaN,NaN,...,2043,2026-06-17 17:12:33.814803,"FloatDistribution(high=0.6, log=False, low=0.4...","FloatDistribution(high=0.001, log=True, low=1e...","FloatDistribution(high=0.98, log=False, low=0....",2043,None,None,None,None
1,c6808b2cd0cb4bd299d4e36298a96c19,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-17 23:12:41.491000+00:00,2026-06-17 23:12:41.505000+00:00,0.697012,NaN,NaN,NaN,...,2074,2026-06-17 17:12:41.490524,"FloatDistribution(high=0.6, log=False, low=0.4...","FloatDistribution(high=0.001, log=True, low=1e...","FloatDistribution(high=0.98, log=False, low=0....",2074,None,None,None,None
2,78381764f4c047b0b0279a878f923263,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-17 23:14:13.517000+00:00,2026-06-17 23:14:13.529000+00:00,0.697004,NaN,NaN,NaN,...,2402,2026-06-17 17:14:13.516183,"FloatDistribution(high=0.6, log=False, low=0.4...","FloatDistribution(high=0.001, log=True, low=1e...","FloatDistribution(high=0.98, log=False, low=0....",2402,None,None,None,None
3,efc4c0f99f08436c9fa441bc4d394923,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-17 23:13:21.630000+00:00,2026-06-17 23:13:21.644000+00:00,0.696919,NaN,NaN,NaN,...,2218,2026-06-17 17:13:21.629197,"FloatDistribution(high=0.6, log=False, low=0.4...","FloatDistribution(high=0.001, log=True, low=1e...","FloatDistribution(high=0.98, log=False, low=0....",2218,None,None,None,None
4,ce862bb223db44d4beb65d33264c765d,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-17 23:14:09.831000+00:00,2026-06-17 23:14:09.842000+00:00,0.696829,NaN,NaN,NaN,...,2388,2026-06-17 17:14:09.829727,"FloatDistribution(high=0.6, log=False, low=0.4...","FloatDistribution(high=0.001, log=True, low=1e...","FloatDistribution(high=0.98, log=False, low=0....",2388,None,None,None,None


In [11]:
best_per_model = (
    runs
    .sort_values("metrics.average_precision", ascending=False)
    .groupby("params.model")
    .head(1)
    .loc[:, [
        "params.model",
        "metrics.average_precision",
        "run_id"
    ]]
)

print(best_per_model)

     params.model  metrics.average_precision                            run_id
0             xgb                   0.697040  6c5042b4d21c4a1bb11d30c685708b5b
2847           rf                   0.686205  5cb82f8537ea4716ad885624b26d893a


In [ ]:

preds = predict_customer_churn(
    pipeline,
    X_test.head(10)
)

print(preds)

In [ ]:
import shap


def generate_shap_explanations(
    pipeline,
    X_raw: pd.DataFrame,
):
    """
    Generates SHAP values from raw data.
    """

    model = pipeline.named_steps["model"]

    X_transformed_df = transform_raw_data(
        pipeline,
        X_raw,
    )

    explainer = shap.TreeExplainer(model)

    shap_values = explainer(
        X_transformed_df
    )

    return (
        explainer,
        shap_values,
        X_transformed_df,
    )

In [ ]:
def plot_global_shap(
    shap_values,
    X_transformed_df,
):
    """
    Global feature importance.
    """

    shap.plots.beeswarm(
        shap_values,
        max_display=20,
    )

In [ ]:
def plot_shap_importance(
    shap_values,
):
    shap.plots.bar(
        shap_values,
        max_display=20,
    )

In [ ]:
def explain_customer(
    shap_values,
    row_idx=0,
):
    shap.plots.waterfall(
        shap_values[row_idx],
        max_display=15,
    )

In [ ]:
pipeline, best_run = (
    interpret.load_best_run_pipeline(
        experiment_name=EXPERIMENT_NAME,
        db_path=DB_PATH,
        order_by_metric="metrics.average_precision DESC",
    )
)

print_model_details(
    best_run,
    pipeline,
)

predictions = predict_customer_churn(
    pipeline,
    X_test.head(10),
)

print(predictions)

(
    explainer,
    shap_values,
    X_transformed_df,
) = generate_shap_explanations(
    pipeline,
    X_test.sample(
        500,
        random_state=42,
    ),
)

plot_shap_importance(
    shap_values
)

plot_global_shap(
    shap_values,
    X_transformed_df,
)

explain_customer(
    shap_values,
    row_idx=0,
)